# **Collaborative Filtering**

In [3]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

Matplotlib is building the font cache; this may take a moment.


In [4]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

For the dataset using "MovieLens-100k dataset" feels right. This dataset basically contains 100,000 ratings from 943 users across 1,682 movies and the ratings being on a scale of 1 to 5.The feed-back matrix has roughly 6.3% density only which is just as almost same as the real world case. This sparsity also helps fitting the data for processing and calculations.Also every user has rated atleats rated 20 movies ensuring memory-based neighbourhood dont fail cuz of empty histories.

In [5]:
def load_movielens_100k():
    data_dir = "./ml-100k"
    data_file = os.path.join(data_dir, "u.data")
    if not os.path.exists(data_file):
        print("downloading")
        url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
        zip_path = "ml-100k.zip"
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(".")
        os.remove(zip_path)
    
    cols = ["user_id", "item_id", "rating", "timestamp"]
    df = pd.read_csv(data_file, sep="\t", names=cols, engine="python")
    
    df["user_idx"] = df["user_id"] - 1
    df["item_idx"] = df["item_id"] - 1
    
    return df

df = load_movielens_100k()
num_users = df["user_idx"].nunique()
num_items = df["item_idx"].nunique()
global_mean = df["rating"].mean()

print(f"Dataset: {len(df)} ratings | {num_users} users | {num_items} items")
print(f"Global Average Rating: {global_mean:.4f}")

train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)
print(f"Train split: {len(train_df)} rows | Test split: {len(test_df)} rows")

downloading
Dataset: 100000 ratings | 943 users | 1682 items
Global Average Rating: 3.5299
Train split: 80000 rows | Test split: 20000 rows
